In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# VİZYON 2027 — v8 FULL KURULUM (Sonsuz Döngüsüz)
# ============================================================
# Bu hücre şunları YAPAR ve BİTİNCE DURUR:
#   1) zstd kurar (Ollama için gerekli)
#   2) Eski process'leri öldürür
#   3) Ollama'yı kurar (yoksa)
#   4) Ollama servisini başlatır
#   5) 13 modeli indirir (eksik olanları)
#   6) Inrok'u kurar + login
#   7) vizyon8 tünelini başlatır
#   8) Local + Public test yapar
#   9) BİTER → Sonsuz döngü YOK (Hücre 2'yi ayrı çalıştıracaksın)
# ============================================================

import subprocess
import time
import os
import json
import shutil
import sys

# ------------------------------------------------------------
# AYARLAR
# ------------------------------------------------------------
VIZYON_NAME = "vizyon8"
VIZYON_URL  = "https://vizyon8.share.inrok.in"

MODELS = [
    "granite3.3:2b", "qwen2.5vl:3b", "phi4-mini-reasoning", "gemma3:4b",
    "qwen3:8b", "qwen3:4b", "qwen2.5-coder", "mistral",
    "phi3.5", "gemma2", "llama3.2", "qwen2.5", "deepseek-r1",
]

OLLAMA_HOST  = "0.0.0.0:11434"
OLLAMA_LOCAL = "http://127.0.0.1:11434"

def log(msg):
    print(msg, flush=True)

# ============================================================
# 0) ZSTD KURULUMU (Ollama için zorunlu)
# ============================================================
log("=" * 70)
log("🚀 VİZYON 2027 — v8 FULL KURULUM BAŞLATILIYOR")
log("=" * 70)

log("\n🔧 zstd kontrol ediliyor...")
zstd_path = shutil.which("zstd")
if zstd_path is None:
    log("⚠️ zstd yok, kuruluyor...")
    subprocess.run(
        "sudo apt-get update -qq && sudo apt-get install -y -qq zstd",
        shell=True, capture_output=True, text=True
    )
    time.sleep(2)
    zstd_path = shutil.which("zstd")
    if zstd_path:
        log(f"✅ zstd kuruldu: {zstd_path}")
    else:
        log("❌ zstd kurulamadı! İnternet açık mı?")
        sys.exit(1)
else:
    log(f"✅ zstd zaten var: {zstd_path}")

# ============================================================
# 1) ESKİ PROCESS'LERİ ÖLDÜR
# ============================================================
log("\n🔄 Eski process'ler öldürülüyor...")
os.system("pkill -f inrok 2>/dev/null")
os.system("pkill -f cloudflared 2>/dev/null")
os.system("pkill -f 'ollama serve' 2>/dev/null")
time.sleep(3)
log("✅ Temizlendi")

# ============================================================
# 2) OLLAMA KURULUMU
# ============================================================
log("\n🦙 Ollama kontrol ediliyor...")
ollama_path = shutil.which("ollama")

if ollama_path is None:
    log("⚠️ Ollama yok, kuruluyor (2-3 dk)...")
    install_proc = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    log("📥 Kurulum çıktısı (son 400 karakter):")
    log((install_proc.stdout or "")[-400:])
    if install_proc.stderr:
        log("⚠️ Hata çıktısı (son 200 karakter):")
        log(install_proc.stderr[-200:])

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for p in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/root/.ollama/bin/ollama"]:
            if os.path.exists(p):
                ollama_path = p
                os.environ["PATH"] = os.path.dirname(p) + ":" + os.environ.get("PATH", "")
                break

    if ollama_path is None:
        log("❌ Ollama kurulamadı!")
        sys.exit(1)
    log(f"✅ Ollama kuruldu: {ollama_path}")
else:
    log(f"✅ Ollama zaten var: {ollama_path}")

# Path garantisi
ollama_dir = os.path.dirname(ollama_path)
if ollama_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = ollama_dir + ":" + os.environ.get("PATH", "")

# ============================================================
# 3) OLLAMA SERVİSİNİ BAŞLAT
# ============================================================
log("\n🦙 Ollama servisi kontrol ediliyor...")
health = subprocess.run(
    f"curl -sf {OLLAMA_LOCAL}/api/tags",
    shell=True, capture_output=True, text=True
)

if health.returncode != 0:
    log("⚠️ Ollama çalışmıyor, başlatılıyor...")
    env = os.environ.copy()
    env["OLLAMA_HOST"] = OLLAMA_HOST
    log_file = open("/kaggle/working/ollama.log", "a", buffering=1)
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=log_file, stderr=subprocess.STDOUT,
        env=env, start_new_session=True
    )
    time.sleep(12)

    health2 = subprocess.run(
        f"curl -sf {OLLAMA_LOCAL}/api/tags",
        shell=True, capture_output=True, text=True
    )
    if health2.returncode == 0:
        log("✅ Ollama başlatıldı")
    else:
        log("⏳ 10 sn daha bekleniyor...")
        time.sleep(10)
        health3 = subprocess.run(
            f"curl -sf {OLLAMA_LOCAL}/api/tags",
            shell=True, capture_output=True, text=True
        )
        if health3.returncode == 0:
            log("✅ Ollama cevap veriyor")
        else:
            log("❌ Ollama başlatılamadı! Log:")
            try: log(open("/kaggle/working/ollama.log").read()[-1500:])
            except: pass
            sys.exit(1)
else:
    log("✅ Ollama zaten çalışıyor")

# ============================================================
# 4) 13 MODELİ KONTROL ET VE EKSİKSE İNDİR
# ============================================================
log("\n🧠 Modeller kontrol ediliyor...")
result = subprocess.run(
    f"{ollama_path} list",
    shell=True, capture_output=True, text=True
)
log("📋 Mevcut modeller:")
log(result.stdout if result.stdout else "(boş)")

existing_lower = (result.stdout or "").lower()
missing_models = []
for m in MODELS:
    base = m.split(":")[0].lower()
    if m.lower() in existing_lower: continue
    if base + ":" in existing_lower: continue
    if base in existing_lower and (base + ":latest") in existing_lower: continue
    missing_models.append(m)

if missing_models:
    log(f"\n📥 İndirilecek: {len(missing_models)} model")
    for m in missing_models: log(f"   • {m}")
    log("\n⏳ Bu işlem 15-30 dakika sürebilir...")
    log("=" * 70)

    for i, m in enumerate(missing_models, 1):
        log(f"\n[{i}/{len(missing_models)}] 📥 {m}")
        t0 = time.time()
        pull_proc = subprocess.run(
            f"{ollama_path} pull {m}",
            shell=True, capture_output=True, text=True
        )
        dt = int(time.time() - t0)
        if pull_proc.returncode == 0:
            log(f"   ✅ {m} indirildi ({dt}s)")
        else:
            log(f"   ⚠️ {m} indirilemedi: {pull_proc.stderr[:200]}")

    log("\n" + "=" * 70)
    log("✅ İndirme tamamlandı")
else:
    log("\n✅ Tüm 13 model zaten hazır")

final_list = subprocess.run(
    f"{ollama_path} list",
    shell=True, capture_output=True, text=True
)
final_count = len([l for l in (final_list.stdout or "").splitlines()[1:] if l.strip()])
log(f"\n🎯 Toplam model: {final_count}")

# ============================================================
# 5) INROK KURULUMU + LOGIN
# ============================================================
log("\n🔐 Inrok kontrol ediliyor...")
inrok_path = shutil.which("inrok")

if inrok_path is None:
    log("⚠️ Inrok yok, kuruluyor...")
    subprocess.run(
        "curl -fsSL https://inrok.in/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    time.sleep(3)
    inrok_path = shutil.which("inrok")
    if inrok_path is None:
        subprocess.run(
            "pip install inrok 2>/dev/null || true",
            shell=True, capture_output=True, text=True
        )
        inrok_path = shutil.which("inrok")

if inrok_path:
    log(f"✅ Inrok hazır: {inrok_path}")
else:
    log("⚠️ Inrok kurulumu başarısız olabilir")

log("\n🔐 Inrok login...")
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    INROK_KEY = secrets.get_secret("INROK_API_KEY")
    login = subprocess.run(
        ["inrok", "login", "--key", INROK_KEY],
        capture_output=True, text=True
    )
    if login.returncode == 0:
        log("✅ Inrok login tamam")
    else:
        log("⚠️ Login zaten yapılmış olabilir")
except Exception as e:
    log(f"⚠️ Secret hatası: {e}")

# ============================================================
# 6) VİZYON8 TÜNELİNİ BAŞLAT
# ============================================================
log(f"\n🚇 Tünel başlatılıyor: {VIZYON_NAME}")
log(f"🌐 URL: {VIZYON_URL}")

inrok_log_file = open("/kaggle/working/inrok8.log", "a", buffering=1)
inrok_bin = inrok_path if inrok_path else "inrok"

tunnel = subprocess.Popen(
    [inrok_bin, "http", "11434", "--name", VIZYON_NAME],
    stdout=inrok_log_file, stderr=subprocess.STDOUT,
    start_new_session=True
)

log("⏳ Tünel yayılıyor (20 sn)...")
time.sleep(20)

if tunnel.poll() is not None:
    log("❌ Tünel başlatılamadı! Log:")
    try: log(open("/kaggle/working/inrok8.log").read()[-2000:])
    except: pass
    sys.exit(1)
else:
    log(f"✅ Tünel çalışıyor (PID: {tunnel.pid})")

# ============================================================
# 7) LOCAL + PUBLIC TEST
# ============================================================
log("\n🧪 Local test...")
local = subprocess.run(
    f"curl -s {OLLAMA_LOCAL}/api/tags",
    shell=True, capture_output=True, text=True
)
if local.returncode == 0 and "models" in (local.stdout or ""):
    try:
        d = json.loads(local.stdout)
        log(f"✅ Local Ollama: {len(d.get('models', []))} model")
    except: log("✅ Local Ollama cevap veriyor")
else:
    log("⚠️ Local test başarısız")

log("\n🌍 Public test (10 sn)...")
time.sleep(10)
try:
    import requests
    headers = {"skip_zrok_interstitial": "true", "User-Agent": "curl/8.0"}
    r = requests.get(VIZYON_URL + "/api/tags", headers=headers, timeout=30)
    log(f"HTTP: {r.status_code}")
    if r.status_code == 200:
        try:
            log(f"✅ Public tünel aktif: {len(r.json().get('models', []))} model")
        except: log("✅ Public tünel aktif")
    else:
        log("⚠️ Henüz hazır değil, 5-10 dk sonra tekrar dene")
except Exception as e:
    log(f"⚠️ Public test: {e}")

# ============================================================
# ✅ HÜCRE 1 BİTTİ
# ============================================================
log("\n" + "=" * 70)
log("✅ HÜCRE 1 TAMAMLANDI — HER ŞEY HAZIR!")
log("=" * 70)
log(f"🌐 Tünel URL: {VIZYON_URL}")
log(f"📊 Model sayısı: {final_count}")
log(f"🔑 SashiDo Config'e yaz: COLAB_LLAMA_URL = {VIZYON_URL}")
log("")
log("👉 ŞİMDİ HÜCRE 2'Yİ ÇALIŞTIR (Sonsuz Döngü + Heartbeat)")
log("=" * 70)

In [ ]:
# ============================================================
# VİZYON 2027 — v8 SONSUZ DÖNGÜ + HEARTBEAT
# ============================================================
# Bu hücre notebook'u sonsuza kadar canlı tutar.
# Hücre 1 zaten tüneli ve modelleri başlattı.
# Bu hücre sadece her 10 dakikada bir kontrol eder.
#
# DURDURMAK İÇİN:
#   - Hücreyi durdur (■) butonuna bas
#   - VEYA: Kernel menüsü → Interrupt
#   - VEYA: Kernel menüsü → Restart
# ============================================================

import subprocess
import time
import threading

OLLAMA_LOCAL = "http://127.0.0.1:11434"
VIZYON_URL   = "https://vizyon8.share.inrok.in"

def log(msg):
    print(msg, flush=True)

log("=" * 70)
log("💓 HEARTBEAT AKTİF — NOTEBOOK AÇIK KALACAK")
log("=" * 70)
log(f"🌐 Tünel URL: {VIZYON_URL}")
log("⚠️ BU HÜCREYİ DURDURMAYIN!")
log("   Durdurursan Kaggle notebook 12 saat sonra kapanır.")
log("=" * 70)

def heartbeat():
    try:
        import requests as _rq
    except Exception:
        _rq = None

    counter = 0
    while True:
        counter += 1
        try:
            if _rq:
                _rq.get(OLLAMA_LOCAL + "/api/tags", timeout=5)
            else:
                subprocess.run(
                    f"curl -s {OLLAMA_LOCAL}/api/tags",
                    shell=True, capture_output=True, timeout=5
                )
            log(f"💓 Heartbeat #{counter}: {time.strftime('%H:%M:%S')} — Ollama OK")
        except Exception as e:
            log(f"⚠️ Heartbeat #{counter}: {time.strftime('%H:%M:%S')} — {e}")
        time.sleep(600)  # 10 dakika

# Heartbeat thread başlat (arka planda çalışır)
threading.Thread(target=heartbeat, daemon=True).start()
log("✅ Heartbeat thread başlatıldı\n")

log("⏳ Sonsuz döngü — Notebook canlı kalıyor...")
log("   (Bu hücreyi durdurana kadar devam eder)\n")

# Sonsuz döngü (ana thread — notebook açık kalır)
while True:
    time.sleep(60)

In [2]:
# ============================================================
# VİZYON 2027 — v9 FULL KURULUM (Sonsuz Döngüsüz)
# ============================================================
# Bu hücre şunları YAPAR ve BİTİNCE DURUR:
#   1) zstd kurar
#   2) Eski process'leri öldürür
#   3) Ollama'yı kurar (yoksa)
#   4) Ollama servisini başlatır
#   5) 13 modeli indirir (eksik olanları)
#   6) Inrok'u kurar + login
#   7) vizyon9 tünelini başlatır
#   8) Local + Public test yapar
#   9) Tünel URL'sini /kaggle/working/tunnel_url.txt dosyasına kaydeder
#  10) BİTER → Sonsuz döngü YOK (Hücre 2 ayrı çalıştırılacak)
# ============================================================

import subprocess
import time
import os
import json
import shutil
import sys
from datetime import datetime

# ------------------------------------------------------------
# AYARLAR
# ------------------------------------------------------------
VIZYON_NAME = "vizyon9"
VIZYON_URL  = "https://vizyon9.share.inrok.in"

MODELS = [
    "granite3.3:2b", "qwen2.5vl:3b", "phi4-mini-reasoning", "gemma3:4b",
    "qwen3:8b", "qwen3:4b", "qwen2.5-coder", "mistral",
    "phi3.5", "gemma2", "llama3.2", "qwen2.5", "deepseek-r1",
]

OLLAMA_HOST  = "0.0.0.0:11434"
OLLAMA_LOCAL = "http://127.0.0.1:11434"

def log(msg):
    print(msg, flush=True)

# ============================================================
# 0) ZSTD KURULUMU
# ============================================================
log("=" * 70)
log("🚀 VİZYON 2027 — v9 FULL KURULUM BAŞLATILIYOR")
log("=" * 70)

log("\n🔧 zstd kontrol ediliyor...")
zstd_path = shutil.which("zstd")
if zstd_path is None:
    log("⚠️ zstd yok, kuruluyor...")
    subprocess.run(
        "sudo apt-get update -qq && sudo apt-get install -y -qq zstd",
        shell=True, capture_output=True, text=True
    )
    time.sleep(2)
    zstd_path = shutil.which("zstd")
    if zstd_path:
        log(f"✅ zstd kuruldu: {zstd_path}")
    else:
        log("❌ zstd kurulamadı! İnternet açık mı?")
        sys.exit(1)
else:
    log(f"✅ zstd zaten var: {zstd_path}")

# ============================================================
# 1) ESKİ PROCESS'LERİ ÖLDÜR
# ============================================================
log("\n🔄 Eski process'ler öldürülüyor...")
os.system("pkill -f inrok 2>/dev/null")
os.system("pkill -f cloudflared 2>/dev/null")
os.system("pkill -f 'ollama serve' 2>/dev/null")
time.sleep(3)
log("✅ Temizlendi")

# ============================================================
# 2) OLLAMA KURULUMU
# ============================================================
log("\n🦙 Ollama kontrol ediliyor...")
ollama_path = shutil.which("ollama")

if ollama_path is None:
    log("⚠️ Ollama yok, kuruluyor (2-3 dk)...")
    install_proc = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    log("📥 Kurulum çıktısı (son 400 karakter):")
    log((install_proc.stdout or "")[-400:])
    if install_proc.stderr:
        log("⚠️ Hata çıktısı (son 200 karakter):")
        log(install_proc.stderr[-200:])

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for p in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/root/.ollama/bin/ollama"]:
            if os.path.exists(p):
                ollama_path = p
                os.environ["PATH"] = os.path.dirname(p) + ":" + os.environ.get("PATH", "")
                break

    if ollama_path is None:
        log("❌ Ollama kurulamadı!")
        sys.exit(1)
    log(f"✅ Ollama kuruldu: {ollama_path}")
else:
    log(f"✅ Ollama zaten var: {ollama_path}")

# Path garantisi
ollama_dir = os.path.dirname(ollama_path)
if ollama_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = ollama_dir + ":" + os.environ.get("PATH", "")

# ============================================================
# 3) OLLAMA SERVİSİNİ BAŞLAT
# ============================================================
log("\n🦙 Ollama servisi kontrol ediliyor...")
health = subprocess.run(
    f"curl -sf {OLLAMA_LOCAL}/api/tags",
    shell=True, capture_output=True, text=True
)

if health.returncode != 0:
    log("⚠️ Ollama çalışmıyor, başlatılıyor...")
    env = os.environ.copy()
    env["OLLAMA_HOST"] = OLLAMA_HOST
    log_file = open("/kaggle/working/ollama.log", "a", buffering=1)
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=log_file, stderr=subprocess.STDOUT,
        env=env, start_new_session=True
    )
    time.sleep(12)

    health2 = subprocess.run(
        f"curl -sf {OLLAMA_LOCAL}/api/tags",
        shell=True, capture_output=True, text=True
    )
    if health2.returncode == 0:
        log("✅ Ollama başlatıldı")
    else:
        log("⏳ 10 sn daha bekleniyor...")
        time.sleep(10)
        health3 = subprocess.run(
            f"curl -sf {OLLAMA_LOCAL}/api/tags",
            shell=True, capture_output=True, text=True
        )
        if health3.returncode == 0:
            log("✅ Ollama cevap veriyor")
        else:
            log("❌ Ollama başlatılamadı! Log:")
            try: log(open("/kaggle/working/ollama.log").read()[-1500:])
            except: pass
            sys.exit(1)
else:
    log("✅ Ollama zaten çalışıyor")

# ============================================================
# 4) 13 MODELİ KONTROL ET VE EKSİKSE İNDİR
# ============================================================
log("\n🧠 Modeller kontrol ediliyor...")
result = subprocess.run(
    f"{ollama_path} list",
    shell=True, capture_output=True, text=True
)
log("📋 Mevcut modeller:")
log(result.stdout if result.stdout else "(boş)")

existing_lower = (result.stdout or "").lower()
missing_models = []
for m in MODELS:
    base = m.split(":")[0].lower()
    if m.lower() in existing_lower: continue
    if base + ":" in existing_lower: continue
    if base in existing_lower and (base + ":latest") in existing_lower: continue
    missing_models.append(m)

if missing_models:
    log(f"\n📥 İndirilecek: {len(missing_models)} model")
    for m in missing_models: log(f"   • {m}")
    log("\n⏳ Bu işlem 15-30 dakika sürebilir...")
    log("=" * 70)

    for i, m in enumerate(missing_models, 1):
        log(f"\n[{i}/{len(missing_models)}] 📥 {m}")
        t0 = time.time()
        pull_proc = subprocess.run(
            f"{ollama_path} pull {m}",
            shell=True, capture_output=True, text=True
        )
        dt = int(time.time() - t0)
        if pull_proc.returncode == 0:
            log(f"   ✅ {m} indirildi ({dt}s)")
        else:
            log(f"   ⚠️ {m} indirilemedi: {pull_proc.stderr[:200]}")

    log("\n" + "=" * 70)
    log("✅ İndirme tamamlandı")
else:
    log("\n✅ Tüm 13 model zaten hazır")

final_list = subprocess.run(
    f"{ollama_path} list",
    shell=True, capture_output=True, text=True
)
final_count = len([l for l in (final_list.stdout or "").splitlines()[1:] if l.strip()])
log(f"\n🎯 Toplam model: {final_count}")

# ============================================================
# 5) INROK KURULUMU + LOGIN
# ============================================================
log("\n🔐 Inrok kontrol ediliyor...")
inrok_path = shutil.which("inrok")

if inrok_path is None:
    log("⚠️ Inrok yok, kuruluyor...")
    subprocess.run(
        "curl -fsSL https://inrok.in/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    time.sleep(3)
    inrok_path = shutil.which("inrok")
    if inrok_path is None:
        subprocess.run(
            "pip install inrok 2>/dev/null || true",
            shell=True, capture_output=True, text=True
        )
        inrok_path = shutil.which("inrok")

if inrok_path:
    log(f"✅ Inrok hazır: {inrok_path}")
else:
    log("⚠️ Inrok kurulumu başarısız olabilir")

log("\n🔐 Inrok login...")
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    INROK_KEY = secrets.get_secret("INROK_API_KEY")
    login = subprocess.run(
        ["inrok", "login", "--key", INROK_KEY],
        capture_output=True, text=True
    )
    if login.returncode == 0:
        log("✅ Inrok login tamam")
    else:
        log("⚠️ Login zaten yapılmış olabilir")
except Exception as e:
    log(f"⚠️ Secret hatası: {e}")

# ============================================================
# 6) VİZYON9 TÜNELİNİ BAŞLAT
# ============================================================
log(f"\n🚇 Tünel başlatılıyor: {VIZYON_NAME}")
log(f"🌐 URL: {VIZYON_URL}")

inrok_log_file = open("/kaggle/working/inrok9.log", "a", buffering=1)
inrok_bin = inrok_path if inrok_path else "inrok"

tunnel = subprocess.Popen(
    [inrok_bin, "http", "11434", "--name", VIZYON_NAME],
    stdout=inrok_log_file, stderr=subprocess.STDOUT,
    start_new_session=True
)

log("⏳ Tünel yayılıyor (20 sn)...")
time.sleep(20)

if tunnel.poll() is not None:
    log("❌ Tünel başlatılamadı! Log:")
    try: log(open("/kaggle/working/inrok9.log").read()[-2000:])
    except: pass
    sys.exit(1)
else:
    log(f"✅ Tünel çalışıyor (PID: {tunnel.pid})")

# ============================================================
# 7) LOCAL + PUBLIC TEST
# ============================================================
log("\n🧪 Local test...")
local = subprocess.run(
    f"curl -s {OLLAMA_LOCAL}/api/tags",
    shell=True, capture_output=True, text=True
)
if local.returncode == 0 and "models" in (local.stdout or ""):
    try:
        d = json.loads(local.stdout)
        log(f"✅ Local Ollama: {len(d.get('models', []))} model")
    except: log("✅ Local Ollama cevap veriyor")
else:
    log("⚠️ Local test başarısız")

log("\n🌍 Public test (10 sn)...")
time.sleep(10)
try:
    import requests
    headers = {"skip_zrok_interstitial": "true", "User-Agent": "curl/8.0"}
    r = requests.get(VIZYON_URL + "/api/tags", headers=headers, timeout=30)
    log(f"HTTP: {r.status_code}")
    if r.status_code == 200:
        try:
            log(f"✅ Public tünel aktif: {len(r.json().get('models', []))} model")
        except: log("✅ Public tünel aktif")
    else:
        log("⚠️ Henüz hazır değil, 5-10 dk sonra tekrar dene")
except Exception as e:
    log(f"⚠️ Public test: {e}")

# ============================================================
# 8) TÜNEL URL'SİNİ DOSYAYA KAYDET (GitHub Actions için)
# ============================================================
log("\n💾 Tünel URL kaydediliyor...")

output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)

# URL dosyası
url_file = os.path.join(output_dir, "tunnel_url.txt")
with open(url_file, "w") as f:
    f.write(VIZYON_URL)
log(f"✅ URL kaydedildi: {url_file}")

# Metadata dosyası (GitHub Actions için)
meta_file = os.path.join(output_dir, "metadata.json")
metadata = {
    "tunnel_url": VIZYON_URL,
    "tunnel_name": VIZYON_NAME,
    "timestamp": datetime.now().isoformat(),
    "model_count": final_count,
    "status": "running",
    "ollama_local": OLLAMA_LOCAL
}
with open(meta_file, "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
log(f"✅ Metadata kaydedildi: {meta_file}")

# ============================================================
# ✅ HÜCRE 1 BİTTİ
# ============================================================
log("\n" + "=" * 70)
log("✅ HÜCRE 1 TAMAMLANDI — HER ŞEY HAZIR!")
log("=" * 70)
log(f"🌐 Tünel URL: {VIZYON_URL}")
log(f"📊 Model sayısı: {final_count}")
log(f"🔑 SashiDo Config: COLAB_LLAMA_URL = {VIZYON_URL}")
log("")
log("📁 Kaydedilen dosyalar:")
log(f"   • {url_file}")
log(f"   • {meta_file}")
log("")
log("👉 ŞİMDİ HÜCRE 2'Yİ ÇALIŞTIR (Sonsuz Döngü)")
log("=" * 70)

🚀 VİZYON 2027 — v9 FULL KURULUM BAŞLATILIYOR

🔧 zstd kontrol ediliyor...
✅ zstd zaten var: /usr/bin/zstd

🔄 Eski process'ler öldürülüyor...
✅ Temizlendi

🦙 Ollama kontrol ediliyor...
✅ Ollama zaten var: /usr/local/bin/ollama

🦙 Ollama servisi kontrol ediliyor...
⚠️ Ollama çalışmıyor, başlatılıyor...
✅ Ollama başlatıldı

🧠 Modeller kontrol ediliyor...
📋 Mevcut modeller:
NAME                          ID              SIZE      MODIFIED      
deepseek-r1:latest            6995872bfe4c    5.2 GB    3 minutes ago    
qwen2.5:latest                845dbda0ea48    4.7 GB    4 minutes ago    
llama3.2:latest               a80c4f17acd5    2.0 GB    4 minutes ago    
gemma2:latest                 ff02c3702f32    5.4 GB    5 minutes ago    
phi3.5:latest                 61819fb370a3    2.2 GB    6 minutes ago    
mistral:latest                6577803aa9a0    4.4 GB    6 minutes ago    
qwen2.5-coder:latest          dae161e27b0e    4.7 GB    6 minutes ago    
qwen3:4b                      359d7dd4b

In [ ]:
# ============================================================
# VİZYON 2027 — v9 SONSUZ DÖNGÜ + HEARTBEAT
# ============================================================
# Bu hücre notebook'u sonsuza kadar canlı tutar.
# Hücre 1 zaten tüneli ve modelleri başlattı.
#
# DURDURMAK İÇİN:
#   - Hücreyi durdur (■)
#   - VEYA: Kernel menüsü → Interrupt
# ============================================================

import subprocess
import time
import threading

OLLAMA_LOCAL = "http://127.0.0.1:11434"
VIZYON_URL   = "https://vizyon9.share.inrok.in"

def log(msg):
    print(msg, flush=True)

log("=" * 70)
log("💓 HEARTBEAT AKTİF — NOTEBOOK AÇIK KALACAK")
log("=" * 70)
log(f"🌐 Tünel URL: {VIZYON_URL}")
log("⚠️ BU HÜCREYİ DURDURMAYIN!")
log("=" * 70)

def heartbeat():
    try:
        import requests as _rq
    except Exception:
        _rq = None

    counter = 0
    while True:
        counter += 1
        try:
            if _rq:
                _rq.get(OLLAMA_LOCAL + "/api/tags", timeout=5)
            else:
                subprocess.run(
                    f"curl -s {OLLAMA_LOCAL}/api/tags",
                    shell=True, capture_output=True, timeout=5
                )
            log(f"💓 Heartbeat #{counter}: {time.strftime('%H:%M:%S')} — Ollama OK")
        except Exception as e:
            log(f"⚠️ Heartbeat #{counter}: {time.strftime('%H:%M:%S')} — {e}")
        time.sleep(600)  # 10 dakika

# Heartbeat thread başlat
threading.Thread(target=heartbeat, daemon=True).start()
log("✅ Heartbeat thread başlatıldı\n")

log("⏳ Sonsuz döngü — Notebook canlı kalıyor...")
log("   (Bu hücreyi durdurana kadar devam eder)\n")

# Sonsuz döngü
while True:
    time.sleep(60)

💓 HEARTBEAT AKTİF — NOTEBOOK AÇIK KALACAK
🌐 Tünel URL: https://vizyon9.share.inrok.in
⚠️ BU HÜCREYİ DURDURMAYIN!
✅ Heartbeat thread başlatıldı

⏳ Sonsuz döngü — Notebook canlı kalıyor...
   (Bu hücreyi durdurana kadar devam eder)

💓 Heartbeat #1: 19:49:07 — Ollama OK
